# Document Field Extraction: OCR vs. Vision-Language Model

Compares an EasyOCR pipeline against a vision-language model (BLIP) for extracting
store names, dates, and totals from real-world receipt photos.

## Setup

In [1]:
!pip install -q easyocr transformers accelerate pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 20.8 MB/s eta 0:00:00


## Load Image Files

Upload your receipt images into a folder named `receipt_images` in this Colab
session before running this cell.

In [2]:
import os

image_folder = "/content/receipt_images"
valid_extensions = (".jpg", ".jpeg", ".png")

image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)])
print(f"Found {len(image_files)} images: {image_files}")

Found 20 images: ['0.jpg', '1.jpg', '10.jpg', '11.jpg', '12.jpg', '13.jpg', '14.jpg', '15.jpg', '16.jpg', '17.jpg', '18.jpg', '19.jpg', '2.jpg', '3.jpg', '4.jpg', '5.jpg', '6.JPG', '7.jpg', '8.jpg', '9.jpg']


## OCR Pipeline

In [3]:
import easyocr

reader = easyocr.Reader(['en'])

results = {}

for filename in image_files:
    path = os.path.join(image_folder, filename)
    text_results = reader.readtext(path, detail=0)
    results[filename] = " ".join(text_results)
    print(f"{filename}: {results[filename][:100]}")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete0.jpg: WALAMART ALWAYS Low PRICES_ Alwane SUPERCENTER OPEN 24 HOURS MANAGER TBA 515 986 1783 ST# 5748 OP# 0
1.jpg: TRADER JOE'$ 2001 Greenvi Ile_Ave Da| as TX 75206 Store #403 (469) 334-0614 OPEN 8:OOAM TO 9:OOPM DA
10.jpg: SPAR Te] 036-4481240 ela 1 bergsparetelkomsa , net VAT  No 4450102696 NLA REG  NO; KZN 032577 LAZENB
11.jpg: WHOLE FOODS fEnR Ai SHARON RD _ TORTILLA" $ CAGE FREE ALL UHIT 3 . 69 BLack BEANS 29 Frozen Hangoes 
12.jpg: WAL MART Save money. Live better. We SELL FOR LESS MANAGEr DapAVE SESSION 636 536 4601 CHESTERFIELD 
13.jpg: I0 # M1 iswuybkcw Walmart Save money. Live better. 8443 21 0362 MANACIR MICHAEL KIGHPD 2014 $ [6Y M 
14.jpg: X" See back of receipt for chance to win $1000 62 ID #: 7LORXAKBPJX ~" 0*' 3 Walmart Save money: Liv
15.jpg: [l6t 9f recelpt for Your Win s1000 pe} TJYYZJVOK3Y Walmart Save money. Live ol 714 799 ~18285 J;'4 h
16.jpg: See back of receipt for your  chance to win 

## Full-Text Review — Total Amount Accuracy Check

Manually inspecting full extracted text for 6 receipts to check total-amount extraction.

In [4]:
check_receipts = ["1.jpg", "2.jpg", "3.jpg", "4.jpg", "9.jpg", "18.jpg"]

for fname in check_receipts:
    print(f"=== {fname} ===")
    print(results[fname])
    print()

=== 1.jpg ===
TRADER JOE'$ 2001 Greenvi Ile_Ave Da| as TX 75206 Store #403 (469) 334-0614 OPEN 8:OOAM TO 9:OOPM DAILY R-CARROTS   SHREDDED 10 0Z ERCUGDESEERUSFESTAU SAEE 8 TOMATOES   CRUSHED TOMAToES WHoLe_No S4LTOW/BASIL ORGANIC" olo_EASHIONED OATMEAL 69 PG S#REDDEDOMGEqESELLA Lite ; EGesshREO_edeGARIcREROHN! 8 8 BEANS ' GARBANZQ #proucebos #ssyLEag 4ct 0 A-APPLE BAG JAZZ 2 LB A-PEPPER BELL  EACH_XL RED 98 GROCERY NON  TAXABLE 0,49 0,87 BANANAS' ORGANIC 3F4 Eo29EENut Butter 2.49 CREAMY SALTED 1.69 WHL' WHT  PITA_BREAD  ,38 GROCERY NON TAXABLE @ 0.69 838.68 SUBTOTAL 838.68 TOTAL 840. 00 CASH 81,32 CHANGE ins Ryan ITEMS 22 12:34PM 0403 04 4398" 4683 06-28-2014 THANK YOU FOR SHOPPING AT TRADER JOE'$ WWw traderjoes.COM

=== 2.jpg ===
Give us feedback survey.wal mart com you! ID #: ZPB65JWCFZ2 Walmart 949 498-6669 Mgr : MICHAEL 951 AVENIDA PICO SAN CLEMENTE CA 92673 ST# 02527 OP# 009045 TE# 45 TR# 06193 GV OATMEAL 007874243408 76 OT  2002 TUM 081236803115 6.74 X M ATHLETICS 019104567781 24

## OCR Accuracy Summary

| Receipt | Result | Failure type |
|---|---|---|
| 1.jpg (Trader Joe's) | Partial | Character misread ($ → 8) |
| 2.jpg (Walmart) | Partial | Field dropped, recovered via duplicate value |
| 3.jpg (Walmart, highlighter) | Failed | Field dropped, unrecoverable |
| 4.jpg (Walmart, blurry) | Failed | Field dropped, unrecoverable |
| 9.jpg (WinCo) | Recovered | Field reordered/relabeled, still correct |
| 18.jpg (Walmart) | Failed | Field dropped, unrecoverable |

Store name recognition was correctly identified across the large majority of the 20
receipts, with two consistent misreads across repeated runs: Costco (stylized logo,
misread as "cexcg") and Walmart ("WALAMART" — the star icon between "WAL" and "MART"
disrupting character segmentation). Both errors reproduced identically on a second
full rerun of the notebook, confirming these are consistent model behaviors rather
than one-off noise.

## Vision-Language Model Comparison

In [5]:
from transformers import BlipProcessor, BlipForQuestionAnswering
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vlm_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to("cuda")

def ask_vlm(image_path, question):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(image, question, return_tensors="pt").to("cuda")
    out = vlm_model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)

vlm_test_images = ["1.jpg", "2.jpg", "3.jpg", "4.jpg"]

for fname in vlm_test_images:
    path = os.path.join(image_folder, fname)
    print(f"=== {fname} ===")
    print("Answer:", ask_vlm(path, "What store and date is on this receipt?"))
    print()

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

=== 1.jpg ===


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Answer: thrift store

=== 2.jpg ===
Answer: walmart

=== 3.jpg ===
Answer: walmart

=== 4.jpg ===
Answer: walmart



## VLM Follow-Up: Single-Focus Question Test

Testing whether question phrasing affects VLM performance — asking about the date
alone, separately from the store name, on the same 4 images.

In [6]:
for fname in vlm_test_images:
    path = os.path.join(image_folder, fname)
    print(f"=== {fname} ===")
    print("Date answer:", ask_vlm(path, "What is the date on this receipt?"))
    print()

=== 1.jpg ===
Date answer: april 17

=== 2.jpg ===
Date answer: may 17

=== 3.jpg ===
Date answer: 7 7

=== 4.jpg ===
Date answer: 17 7 2012



## Conclusion

OCR substantially outperformed the vision-language model for extracting precise
document text. On store name recognition, BLIP performed reasonably (3 of 4 correct),
suggesting it can recognize coarse visual patterns like branding or logo style.
However, on date extraction — tested both as part of a compound question and asked
in isolation — BLIP was incorrect on all 4 images regardless of phrasing, producing
plausible-looking but fabricated dates (e.g., "april 17," "17 7 2012").

This indicates the gap isn't primarily about question structure, but about a genuine
capability difference: BLIP-vqa-base, trained for general scene understanding, can
approximate coarse visual categories but lacks the fine-grained text-reading
precision needed for small printed fields like dates. OCR, purpose-built for
character-level recognition, is the correct tool for structured document field
extraction — a general-purpose VQA model can supplement coarse classification tasks
but is not a reliable substitute for reading precise text fields.